# Streamlined Building Unit Prediction
Generalizing building unit regression calculations so they can be reproducible across all California data.

### Geospatial Data Sources:
- Building parquet file ([Google Earth Engine](https://sat-io.earthengine.app/view/gba))
- Zillow points (from advisor – not public)
- Residential parcels ([ArcGIS Hub, County of Los Angeles](https://hub.arcgis.com/documents/baaf8251bfb94d3984fb58cb5fd93258/about))

For a more descriptive overview of this process, see the [original notebook](https://github.com/sofiasarak/building-unit-regression/blob/main/unit_regression_la.ipynb).

In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
# from sklearn.linear_model import LinearRegression

## Loading in Data

### Parcel

In [2]:
# load parcels 
parcels = gpd.read_file("/../../capstone/electrigrid/data/parcels/Parcels_CA_2014.gdb",
     layer="CA_PARCELS_STATEWIDE",
      where="County='Ventura'").to_crs(epsg=4326)

## Zillow

In [3]:
# import Zillow data (make take 10-20 minutes)
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

## Building Footprints

In [4]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
fp = "../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet"
building = gpd.read_parquet(fp).to_crs(epsg=4326)

# Building Unit Prediction Analysis

Reminders:
- the Zillow and buildings dataframes contains data for ALL of California, whereas parcels are county-specific 

Assumptions:
- `type` is accurate even when `type` is single and the `unit` is more than 1
- `unit` is accurate even when it doesn't match up with `code`

## Building Linear Model

The model will be trained using multi-family homes that directly intersect with Zillow points.

In [7]:
## FINDING ALL MULTI-FAMILY BUILDINGS BY JOINING ZILLOW -> PARCEL, THEN PARCEL -> BUILDINGS

# select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

In [8]:
## crop only to residential parcels
# keep the indices where multi-family homes match to parcels (.index.unique() de-duplicates)
valid_parcels = parcels.sjoin(zillow_multi, how = "inner", predicate="intersects")[parcels.columns].index.unique()


In [9]:

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

In [15]:
parcels_res

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry
5,073014509,Ventura,None,None,None,182.902643,1858.745127,"MULTIPOLYGON Z (((-119.28341 34.27864 0.00000,..."
6,073015209,Ventura,None,None,None,152.388459,929.205443,"MULTIPOLYGON Z (((-119.28156 34.27892 0.00000,..."
40,074012606,Ventura,None,None,None,114.548318,786.709364,"MULTIPOLYGON Z (((-119.25682 34.27777 0.00000,..."
47,074016115,Ventura,None,None,None,107.308424,646.866396,"MULTIPOLYGON Z (((-119.25479 34.27749 0.00000,..."
52,073018318,Ventura,None,None,None,88.395441,418.262974,"MULTIPOLYGON Z (((-119.27589 34.27719 0.00000,..."
...,...,...,...,...,...,...,...,...
258524,625008129,Ventura,None,None,None,131.768939,998.145600,"MULTIPOLYGON Z (((-118.70708 34.28056 0.00000,..."
258529,526018001,Ventura,None,None,None,935.683387,49689.016129,"MULTIPOLYGON Z (((-118.91884 34.18754 0.00000,..."
258554,101030209,Ventura,None,None,None,116.900737,766.159645,"MULTIPOLYGON Z (((-119.05655 34.36665 0.00000,..."
258757,079035202,Ventura,None,None,None,46.056348,104.842916,"MULTIPOLYGON Z (((-119.24128 34.26694 0.00000,..."


Change the code to keep the `PARNO` (parcel number with each of the buildings).

In [ ]:
# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

In [17]:
buildings_res

,source,id,height,var,region,bbox,geometry
1919731,osm,1263382221,1.689560,0.362742,USA,"{'xmax': -118.79252969999997, 'xmin': -118.792...","POLYGON ((-118.79274 34.40812, -118.79253 34.4..."
1919731,osm,995401202,6.066593,0.386094,USA,"{'xmax': -112.405666, 'xmin': -112.405811, 'ym...","POLYGON ((-112.40581 33.44536, -112.40572 33.4..."
1919731,osm,300871590,5.128046,1.171719,USA,"{'xmax': -122.0018053, 'xmin': -122.0019842999...","POLYGON ((-122.00197 37.39080, -122.00181 37.3..."
1919731,ms,UnitedStates_023013100_6329,2.233225,0.239022,USA,"{'xmax': -115.08982803309591, 'xmin': -115.090...","POLYGON ((-115.08993 36.27611, -115.08983 36.2..."
1919732,osm,1263382226,2.533644,0.326330,USA,"{'xmax': -118.79253410000001, 'xmin': -118.792...","POLYGON ((-118.79272 34.40808, -118.79272 34.4..."
...,...,...,...,...,...,...,...
3714893,ms,UnitedStates_023010033_102697,5.106846,0.463753,USA,"{'xmax': -121.07042611867162, 'xmin': -121.070...","POLYGON ((-121.07043 38.93292, -121.07043 38.9..."
3714895,ms,UnitedStates_023012310_106199,2.706650,1.760612,USA,"{'xmax': -119.27047200582479, 'xmin': -119.270...","POLYGON ((-119.27060 34.25707, -119.27058 34.2..."
3714895,ms,UnitedStates_023010033_100882,5.040120,0.465710,USA,"{'xmax': -121.07041892046726, 'xmin': -121.070...","POLYGON ((-121.07046 38.93347, -121.07060 38.9..."
3714896,ms,UnitedStates_023012310_134972,2.990292,1.289581,USA,"{'xmax': -119.27059842703561, 'xmin': -119.270...","POLYGON ((-119.27071 34.25687, -119.27060 34.2..."


In [ ]:


## CALCULATING VOLUME FOR BOTH SINGLE AND MULTI-FAMILY HOMES

# keep all residential buildings, and add zillow points only where they match up (MULTI)
building_zillow_multi = gpd.sjoin(
    buildings_res,
    zillow_multi,
    how = "left",
    predicate = "intersects")

# filter for only single and condo units
zillow_single = zillow[(zillow['type'] == "Single") | (zillow['code'] == "RR106")]

# keep all residential buildings, and add zillow points only where they match up (SINGLE & CONDOS)
building_zillow_single = gpd.sjoin(
    building,
    zillow_single,
    how = "inner",
    predicate = "intersects")

# combine all for volume calculation
building_zillow_all = pd.concat([building_zillow_multi, building_zillow_single])

# reproject data frame to crs with meters as units
building_m = building_zillow_all.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# save single and condos as its own df
non_multi = building_m[(building_m['type'] == "Single") | (building_m['code'] == "RR106")]

# now that single unit homes also have volume data, we can drop them
building_m = building_m[building_m['type'] == "Multi"]
building_m = building_m[building_m['code'] != "RR106"]

## BUILDING REGRESSION TO PREDICT UNIT DATA WHERE IT IS MISSING

# keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

# run model
results = smf.ols('unit ~ volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('unit ~ volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

## Predicting Units for Multi-Family Homes

We will select all residential multi-family homes by joining building footprints (`buildings_res`) and multi-family Zillow data (`zillow_multi`) on the parcel column. Then, the unit column will be dropped, to ensure that the number of units is not repeated for every building in the parcel. New unit information will be predicted using our model.

In [10]:
# join zillow point to parcels
zillow_w_par = zillow_multi.sjoin(parcels_res, predicate="within")

# join buildings to parcels
building_w_par = buildings_res.sjoin(parcels_res, predicate="within")

# join the two based on parcel number
building_w_zillow = pd.merge(zillow_w_par, building_w_par, on = 'PARNO')
building_w_zillow = building_w_zillow.drop('unit', axis = 1)

# clean columns
building_w_zillow = building_w_zillow.loc[:, "type":"code"].join(building_w_zillow.loc[:, "id":"geometry_y"])

# to gdf
building_w_zillow = gpd.GeoDataFrame(building_w_zillow, geometry="geometry_y", crs="EPSG:4326")

In [11]:
# calculate volume
# reproject data frame to crs with meters as units
building_w_zillow_m = building_w_zillow.to_crs("EPSG:6933")

# create column from polygon area
building_w_zillow_m['area_m2'] = building_w_zillow_m.geometry.area

# rename height column to be clear about units
building_w_zillow_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_w_zillow_m['volume_m3'] = building_w_zillow_m['area_m2'] * building_w_zillow_m['height_m']

In [12]:
building_w_units['residual']

1931048    -3.649276
1931049    -4.484293
1931534     7.360542
1936472    30.574382
1936493    -2.748838
             ...    
3714068    -1.251802
3714197    -3.672458
3714199    -3.930766
3714204   -10.231515
3714205    -5.859025
Name: residual, Length: 186, dtype: float64

In [13]:
# predict units based on volume
building_w_zillow_m['unit'] = round(intercept + building_w_zillow_m['volume_m3'] * slope)

## UNIT PREDICTION COMPLETE

# find centroid of footprints
building_w_zillow_m['centroid'] = building_w_zillow_m.geometry.centroid

# set as geometry
building_w_zillow_n = gpd.GeoDataFrame(building_w_zillow_m, geometry="centroid", crs="EPSG:4326")

## Concatenating with Single-Family Homes

Now that both our single and multi family homes contain points as their geomtries, we can combine them into one, and save.

In [ ]:
# pre-join cleaning
non_multi = non_multi.to_crs(zillow.crs).drop('index_right', axis = 1)

# join Zillow data to non-multi family homes (takes ~1 minute)
non_multi_points = gpd.sjoin(
    zillow, # left df's geometry is always kept
    non_multi,
    how = "inner",
    predicate = "intersects")



In [17]:
non_multi_points.head()

,type_left,year_left,room_left,heat_left,cool_left,own_left,unit_left,value_left,sqft_type_left,sqft_left,ID_left,GEOID_left,p_ID_left,area_left,code_left,geometry,index_right,source,id,height_m,var,region,bbox,type_right,year_right,room_right,heat_right,cool_right,own_right,unit_right,value_right,sqft_type_right,sqft_right,ID_right,GEOID_right,p_ID_right,area_right,code_right,area_m2,volume_m3
0,Multi,2003.0,1.0,None,None,I,224.0,491943.0,living,1003.0,3,06001403302,468,PGE/SCE,RR106,POINT (-122.26800 37.79429),2833893,osm,5995281,24.631056,23.205521,USA,"{'xmax': -122.2673779, 'xmin': -122.2686312, '...",Multi,2003.0,1.0,None,None,None,224.0,155077.0,living,722.0,2362.0,06001403302,468,PGE/SCE,RR106,4953.551609,122011.206244
0,Multi,2003.0,1.0,None,None,I,224.0,491943.0,living,1003.0,3,06001403302,468,PGE/SCE,RR106,POINT (-122.26800 37.79429),2833893,osm,5995281,24.631056,23.205521,USA,"{'xmax': -122.2673779, 'xmin': -122.2686312, '...",Multi,2003.0,1.0,None,None,None,224.0,491943.0,living,1123.0,20.0,06001403302,468,PGE/SCE,RR106,4953.551609,122011.206244
0,Multi,2003.0,1.0,None,None,I,224.0,491943.0,living,1003.0,3,06001403302,468,PGE/SCE,RR106,POINT (-122.26800 37.79429),2833893,osm,5995281,24.631056,23.205521,USA,"{'xmax': -122.2673779, 'xmin': -122.2686312, '...",Multi,2003.0,2.0,None,None,I,224.0,894743.0,living,1586.0,2308.0,06001403302,468,PGE/SCE,RR106,4953.551609,122011.206244
0,Multi,2003.0,1.0,None,None,I,224.0,491943.0,living,1003.0,3,06001403302,468,PGE/SCE,RR106,POINT (-122.26800 37.79429),2833893,osm,5995281,24.631056,23.205521,USA,"{'xmax': -122.2673779, 'xmin': -122.2686312, '...",Multi,2003.0,2.0,None,None,O,224.0,356131.0,living,1540.0,2295.0,06001403302,468,PGE/SCE,RR106,4953.551609,122011.206244
0,Multi,2003.0,1.0,None,None,I,224.0,491943.0,living,1003.0,3,06001403302,468,PGE/SCE,RR106,POINT (-122.26800 37.79429),2833893,osm,5995281,24.631056,23.205521,USA,"{'xmax': -122.2673779, 'xmin': -122.2686312, '...",Multi,2003.0,1.0,None,None,I,224.0,754290.0,living,1089.0,2311.0,06001403302,468,PGE/SCE,RR106,4953.551609,122011.206244


The `non_multi_points` has only `type_left` and `type_right`. Change the code to be `type_left`.

In [18]:

# this join introduces multi-family homes that overlap single-family points
non_multi_points = non_multi_points[non_multi_points['type_left'] != "Multi"]

In [19]:
# selecting only necessary columns
multi = building_w_zillow_m[['type', 'year', 'room', 'heat', 'cool', 'own', 'value', 'sqft_type', 'ID', 'GEOID', 'p_ID', 'code', 'area_m2', 'volume_m3', 'unit', 'centroid']]
multi = multi.rename(columns = {"centroid":"geometry"})

# renaming
non_multi_points.columns = non_multi_points.columns.str.replace("_left", "")
single = non_multi_points[['type', 'year', 'room', 'heat', 'cool', 'own', 'value', 'sqft_type', 'ID', 'GEOID', 'p_ID', 'code', 'area_m2', 'volume_m3', 'unit', 'geometry']]

In [ ]:
## multi doesnt have an active geometry
# set the geometry to active
multi = multi.set_geometry('geometry')
multi.crs

In [20]:
# concatenate multi and single family homes
ventura_building = pd.concat([single, multi]).to_crs(zillow.crs)

ValueError: Cannot determine common CRS for concatenation inputs, got ['WGS 84', 'WGS 84 / NSIDC EASE-Grid 2.0 Global']. Use `to_crs()` to transform geometries to the same CRS before merging.

In [ ]:
ventura_building.to_file("data/ventura_buildings.geojson", driver='GeoJSON')

In [ ]:
ventura_building['type'].unique()

In [ ]:
single['type'].value_counts()

In [ ]:
ventura_building.plot(column = "type")

## Old code:

In [ ]:
## FINDING ALL MULTI-FAMILY BUILDINGS BY JOINING ZILLOW -> PARCEL, THEN PARCEL -> BUILDINGS

# select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels (.index.unique() de-duplicates)
valid_parcels = parcels.sjoin(zillow_multi, how = "inner", predicate="intersects")[parcels.columns].index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
#units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
#parcels_res = parcels_res.join(units_sum)



## CALCULATING VOLUME FOR BOTH SINGLE AND MULTI-FAMILY HOMES

# keep all residential buildings, and add zillow points only where they match up (MULTI)
building_zillow_multi = gpd.sjoin(
    buildings_res,
    zillow_multi,
    how = "left",
    predicate = "intersects")

# filter for only single and condo units
zillow_single = zillow[(zillow['type'] == "Single") | (zillow['code'] == "RR106")]

# keep all residential buildings, and add zillow points only where they match up (SINGLE & CONDOS)
building_zillow_single = gpd.sjoin(
    building,
    zillow_single,
    how = "inner",
    predicate = "intersects")

# combine all for volume calculation
building_zillow_all = pd.concat([building_zillow_multi, building_zillow_single])

# reproject data frame to crs with meters as units
building_m = building_zillow_all.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# save single and condos as its own df
non_multi = building_m[(building_m['type'] == "Single") | (building_m['code'] == "RR106")]

# now that single unit homes also have volume data, we can drop them
building_m = building_m[building_m['type'] == "Multi"]
building_m = building_m[building_m['code'] != "RR106"]



## BUILDING REGRESSION TO PREDICT UNIT DATA WHERE IT IS MISSING

# keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

# run model
results = smf.ols('unit ~ volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('unit ~ volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

# extract just the multi-family homes where unit info is missing
missing_units = building_m[building_m['unit'].isna()]

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['unit'] = round(intercept + missing_outlier_units_pred['volume_m3'] * slope)

# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop excess columns
multi_complete = multi_complete.drop(['residual'], axis = 1)
multi_complete = multi_complete.drop(['index_right'], axis = 1) 

## UNIT PREDICTION COMPLETE



## AGGREGATE BY PARCEL (SUM MULTI-HOME UNITS)

# find the parcels that contain multi-family homes
multi_by_parcel = parcels_res.sjoin(multi_complete, predicate="intersects")

# when aggregated to parcel, there should be less multi-family home observations
#assert len(multi_by_parcel) < len(multi_complete)

# join complete multi-family homes data frame to parcels, then sum units by parcel
summed_units = parcels_res.sjoin(multi_complete, predicate="intersects").groupby(["PARNO"])['unit'].sum()

assert len(summed_units) < len(multi_by_parcel) # this should hold because multi_by_parcel has rows for each building, 
                                                # whereas multi_summed_units is by parcel (and there are less parcels than buildings)

# join unit sums to the parcel geometries themselves, keeping only where units were summed
multi_summed_units = parcels_res.join(summed_units).dropna(subset = ['unit'])

assert len(multi_summed_units) < len(multi_by_parcel)

## SAVING NON-MULTI-FAMILY (SINGLE AND CONDO) OBSERVATIONS

non_multi = non_multi[multi_complete.columns].to_crs(zillow.crs)

# keep only variables of interest
# non_multi = non_multi[['type', 'source', 'id', 'height_m', 'var', 'region', 'bbox', 'geometry', 'area_m2', 'volume_m3']].to_crs(zillow.crs)

# join Zillow data to non-multi family homes (takes ~1 minute)
non_multi_points = gpd.sjoin(
    zillow, # left df's geometry is always kept
    non_multi,
    how = "inner",
    predicate = "intersects")

In [ ]:
# concatenate multi and single family homes
ventura_building = pd.concat([non_multi, multi_complete]).to_crs(zillow.crs)

In [ ]:
# non_multi_points_test.to_file("data/non_multi_zillow_test.geojson", driver='GeoJSON')

In [ ]:
# multi_summed_units_test.to_file("data/multi_summed_units_test.geojson", driver='GeoJSON')